# Regression tables with MakeTables

[MakeTables](https://py-econometrics.github.io/maketables/) creates formatted regression tables from fitted DoubleML models. This example uses simulated data to show model comparisons, confidence intervals, multiple treatments, and HTML and LaTeX exports.

This example requires the DoubleML development version with MakeTables support and `maketables>=0.2.0`. Install MakeTables separately with `python -m pip install maketables`. DoubleML itself does not require MakeTables.

The integration applies to models inheriting from `DoubleML`, including `DoubleMLPLR`, `DoubleMLPLIV`, and `DoubleMLIRM`. It does not yet cover standalone result containers such as `DoubleMLFramework` or `DoubleMLBLP`.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from maketables import ETable
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge

import doubleml as dml

np.random.seed(42)

## Fit a partially linear model

The simulated outcome depends on two treatments and three covariates. The treatment coefficients are 0.6 and 0.4. We estimate both effects with linear nuisance models and three repetitions of two-fold cross-fitting.

In [2]:
n_obs = 500
x = np.random.normal(size=(n_obs, 3))
education = 0.5 * x[:, 0] + np.random.normal(size=n_obs)
experience = 0.3 * x[:, 1] + np.random.normal(size=n_obs)
income = 0.6 * education + 0.4 * experience + x[:, 2] + np.random.normal(size=n_obs)
frame = pd.DataFrame(x, columns=["x1", "x2", "x3"])
frame["education"] = education
frame["experience"] = experience
frame["income"] = income

data = dml.DoubleMLData(frame, "income", ["education", "experience"])
linear = dml.DoubleMLPLR(data, LinearRegression(), LinearRegression(), n_folds=2, n_rep=3).fit()
linear.summary

,coef,std err,t,P>|t|,2.5 %,97.5 %
education,0.607341,0.041058,14.792432,1.639224e-49,0.526869,0.687812
experience,0.376111,0.045407,8.283145,1.199641e-16,0.287115,0.465107


## Create a table

Pass the fitted model directly to `ETable`. The `coef_fmt` argument controls which statistics appear: `b` is the estimate, `se` its standard error, `t` its t-statistic, and `p` its p-value. An asterisk requests significance stars.

Each row is named after the treatment and the quantity estimated, here `education (PLR)` and `experience (PLR)`. Rows from models that estimate different quantities, such as an ATE and an ATTE, therefore stay separate. To relabel rows, use these full names as keys in `labels`.

Below the estimates, the table reports the number of observations, the standard-error type (`hetero`, or `by: <cluster variables>` for clustered data), the number of folds, and the number of sample-split repetitions. With more than one repetition, the estimate is the median across repetitions. A `Clusters` row is added for clustered data.

Fit models before creating tables. An unfitted model has no coefficient rows, just as its DoubleML summary has no estimates.

In [3]:
labels = {"education (PLR)": "Education", "experience (PLR)": "Experience"}
table = ETable(linear, coef_fmt="b:.3f*\n(se:.3f)", labels=labels)
display(HTML(table.make("html")))

## Compare nuisance learners

Use the same sample splits to compare linear and ridge nuisance models. Each model becomes a column; both treatments appear as rows.

In [4]:
ridge = dml.DoubleMLPLR(data, Ridge(alpha=10), Ridge(alpha=10), n_folds=2, n_rep=3, draw_sample_splitting=False)
ridge.set_sample_splitting(linear.smpls)
ridge.fit()
comparison = ETable(
    [linear, ridge],
    labels=labels,
    model_heads=["Linear", "Ridge"],
    coef_fmt="b:.3f*\n(se:.3f)",
    caption="Treatment effects with different nuisance learners",
)
display(HTML(comparison.make("html")))

## Show confidence intervals

The tokens `ci95l` and `ci95u` contain the bounds returned by DoubleML's pointwise 95% confidence intervals. For repeated sample splits, these use DoubleML's aggregation procedure. They are not simultaneous confidence intervals across treatments.

In [5]:
interval_table = ETable(
    linear,
    labels=labels,
    coef_fmt="b:.3f\n[ci95l:.3f, ci95u:.3f]",
    notes="Pointwise 95% confidence intervals in brackets.",
)
display(HTML(interval_table.make("html")))

After `bootstrap()`, the coefficient table also contains joint 95% bounds, `ci95l_joint` and `ci95u_joint`, which cover both treatments simultaneously. They are wider than the pointwise intervals above; with a single treatment the two are essentially the same.

In [6]:
linear.bootstrap(n_rep_boot=1000)
joint_table = ETable(
    linear,
    labels=labels,
    coef_fmt="b:.3f\n[ci95l_joint:.3f, ci95u_joint:.3f]",
    notes="Joint 95% confidence intervals in brackets.",
)
display(HTML(joint_table.make("html")))

## Binary treatments

The same interface works with `DoubleMLIRM`. Here the simulated treatment effect is 0.7, and treatment assignment depends on the covariates. Estimating the average treatment effect (ATE) and the average treatment effect on the treated (ATTE) on the same data gives two rows, `d (ATE)` and `d (ATTE)`, so the table shows which quantity each column holds.

In [7]:
propensity = 1 / (1 + np.exp(-0.5 * x[:, 0]))
treatment = np.random.binomial(1, propensity)
outcome = 0.7 * treatment + x[:, 2] + np.random.normal(size=n_obs)
irm_data = dml.DoubleMLData.from_arrays(x, outcome, treatment)
irm = dml.DoubleMLIRM(irm_data, LinearRegression(), LogisticRegression(), n_folds=2, n_rep=3).fit()
irm_atte = dml.DoubleMLIRM(irm_data, LinearRegression(), LogisticRegression(), score="ATTE", n_folds=2, n_rep=3).fit()
display(HTML(ETable([irm, irm_atte], coef_fmt="b:.3f*\n(se:.3f)").make("html")))

## Export tables

`make("tex")` returns LaTeX source. To write files, use `table.save("tex", "results.tex")` or `table.save("html", "results.html")`. See the [MakeTables LaTeX guide](https://py-econometrics.github.io/maketables/docs/LatexTables.html) for document setup.

In [8]:
print(table.make("tex"))

\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{1}{c}{income} \\
\cmidrule(lr){2-2}
 & (1) \\
\midrule
\addlinespace[1ex]
Education & \makecell{0.607***\\(0.041)} \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
Experience & \makecell{0.376***\\(0.045)} \\
\addlinespace[0.5ex]
\midrule
\addlinespace[1ex]
Observations & 500 \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
S.E. type & hetero \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
Folds & 2 \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
Sample-split repetitions & 3 (median) \\
\addlinespace[0.5ex]
\bottomrule
\end{tabularx}
\endgroup
\noindent\begin{minipage}{\linewidth}\smallskip\footnotesize
Significance levels: * p < 0.1, ** p < 0.05, *** p < 0.01. Format of coefficient cell: Coefficient (Std. Error)\end{minipage}

\end{threeparttable}
